# Motif-scaffolding **success predictor** & method recommender

Data-driven rulebook: given a freshly extracted interface motif, predict its **designability per
generator**, decompose *which motif feature* drove the prediction, weigh it against each method's
**compute cost** (ROI = GPU-hours per designable design), and recommend the cheapest route to a
lab-testable design — including whether **filling gaps** (reducing disjointedness) is worth it.

**Model:** binomial **logistic regression** (the simplest thing that explains the data; a plain
logistic was not beaten by fancier combos for design-success prediction, Garcia et al. Prot. Sci.
2026). Built up incrementally (disjointedness → +length → +interaction → +SSE,RSA). Per-feature
contribution shown **nomogram-style** (log-odds decomposition). A shallow decision tree gives the
at-a-glance rulebook. Fit on the **baseline run** `lib_interface_pad5` (+ La-Proteina unindexed).

Run in the `gen_env` kernel. Reading the six score CSVs is the only slow step (fast on the cluster's
local FS; minutes over the sshfs mount).

In [ ]:
import os, json, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
try:
    import statsmodels.api as sm            # nicer CIs / p-values / AIC if present
    HAVE_SM = True
except Exception:
    HAVE_SM = False
print("statsmodels:", HAVE_SM)

LIB   = '/cluster/work/wicky/bekessel/motif_library'
RUN   = f'{LIB}/runs/lib_interface_pad5'
UIDX  = f'{LIB}/runs/lib_interface_pad5_uidx'
CSCALE= f'{LIB}/runs/lib_interface_pad5_contigscale'
POOL  = f'{LIB}/motif_pool_interface_j0.7_pad5.csv'
OUTDIR= f'{LIB}/analysis'; os.makedirs(OUTDIR, exist_ok=True)

# notebook-method -> (run_dir, method_subdir). Baseline 5 + La-Proteina unindexed (best predictor).
LOAD = {'rfdiff':(RUN,'rfdiff'), 'proteina':(RUN,'proteina'), 'genie3':(RUN,'genie3'),
        'pp1c_bb':(RUN,'pp1c_bb'), 'laproteina':(RUN,'laproteina'),
        'laproteina_unindexed':(UIDX,'laproteina')}
LABELS = {'rfdiff':'RFdiffusion','proteina':'Proteina','genie3':'Genie3','pp1c_bb':'Protpardelle',
          'laproteina':'La-Proteina idx','laproteina_unindexed':'La-Proteina uidx'}
PALETTE= {'rfdiff':'#2ec4b6','proteina':'#3b82f6','genie3':'#9b5de5','pp1c_bb':'#ff9f1c',
          'laproteina':'#e71d36','laproteina_unindexed':'#3a86ff'}

MOTIF_RMSD_MAX, SC_RMSD_MAX, RANK_MIN = 1.0, 2.0, 0.75   # the 3 gates
NSAMPLES   = 100      # backbones generated per motif (generation denominator basis)
SEQS_PER   = 8        # MPNN seqs per backbone
AF3_SEC    = 20       # ~seconds per AF3 fold  -> flat shared refold cost, TUNABLE
REFOLD_GPUH_PER_DESIGN = SEQS_PER * AF3_SEC / 3600.0      # GPU-h to refold one backbone's 8 seqs

CONT = ['disjointed_parts','motif_length','mean_rsa']     # continuous features (start set)
def prettify(ax):
    ax.grid(False)
    for s in ax.spines.values(): s.set_visible(True); s.set_edgecolor('black'); s.set_linewidth(1.0)

## 1 · Assemble the per-(motif, method) table
One row per motif × method: post-generation `n_designs`, 3-gate `n_des`, `succ_rate`, motif features, method cost.

In [ ]:
# --- per-(motif,method): designs & 3-gate designable, merged with motif features ---
pool = pd.read_csv(POOL); pool['motif_num'] = pool.motif_num.astype(int)
pool['task_name'] = pool.pdb_name + '_m' + pool.motif_num.astype(str) + '_' + pool.variant
tset = set(pool.task_name)
def to_task(dn):
    p = str(dn).split('_')
    for k in range(len(p), 0, -1):
        c = '_'.join(p[:k])
        if c in tset: return c
    return '_'.join(p[:-1])

frames = []
for m,(run,sub) in LOAD.items():
    f = f'{run}/{sub}/af3_refold_scores.csv'
    if not os.path.exists(f): print('  MISSING', m, f); continue
    d = pd.read_csv(f, usecols=lambda c: c in {'design_name','motifRMSD','scRMSD','ranking_score'})
    d['sp'] = ((d.motifRMSD<=MOTIF_RMSD_MAX)&(d.scRMSD<=SC_RMSD_MAX)&(d.ranking_score>=RANK_MIN)).astype(int)
    g = d.groupby('design_name').sp.max().reset_index()            # design designable = any of its <=8 seqs
    g['task'] = g.design_name.map(to_task)
    t = g.groupby('task').agg(n_designs=('sp','size'), n_des=('sp','sum')).reset_index()
    t['method'] = m; frames.append(t)
    print(f'  {m:22s} designs={len(g):6d} tasks={t.task.nunique():4d} succ={g.sp.mean():.3f}')
tab = pd.concat(frames, ignore_index=True)
tab['succ_rate'] = tab.n_des / tab.n_designs
feat = pool[['task_name','pdb_name','variant','motif_length','disjointed_parts','dominant_sse',
             'mean_rsa','rg','linearity','planarity','mean_cx']]
tab = tab.merge(feat, left_on='task', right_on='task_name', how='left')

# per-method GENERATION cost (GPU-h / design), RUN-AWARE (uidx has its own compute_costs) + flat refold
COSTS = {}
for run in {LOAD[m][0] for m in LOAD}:
    cf = f'{run}/compute_costs.csv'
    if os.path.exists(cf):
        gg = pd.read_csv(cf); gg = gg[gg.stage=='generation'].set_index('method')
        for sub, r in gg.iterrows():
            COSTS[(run, sub)] = r.gpu_hours / max(1, r.n_tasks*NSAMPLES)
def gpuh_per_design(method):
    base = COSTS.get(LOAD[method], np.nan)
    return base + REFOLD_GPUH_PER_DESIGN
tab['gpuh_per_design'] = tab.method.map(gpuh_per_design)
print('\nrows', len(tab), '| unmatched feats', round(tab.motif_length.isna().mean(),3))
tab = tab.dropna(subset=['motif_length'])
display(tab.groupby('method').agg(tasks=('task','nunique'), succ=('succ_rate','mean'),
        gpuh_per_design=('gpuh_per_design','first')).round(3))

In [ ]:
# --- standardize continuous features on the UNIQUE motif set; build design-matrix helper ---
u = tab.drop_duplicates('task')
MU = {c: float(u[c].mean()) for c in CONT}
SD = {c: float(u[c].std(ddof=0)) for c in CONT}
def add_design_cols(df):
    o = df.copy()
    for c in CONT: o[c+'_z'] = (o[c]-MU[c])/SD[c]
    o['H'] = (o.dominant_sse=='H').astype(float)          # SSE dummies, coil = reference
    o['E'] = (o.dominant_sse=='E').astype(float)
    o['len_x_disj'] = o['motif_length_z']*o['disjointed_parts_z']    # the interaction
    return o
tab = add_design_cols(tab)

# multicollinearity: length <-> disjointedness are correlated -> why we standardize + add interaction
corr = u[CONT].corr().round(2)
print('feature correlation (unique motifs):'); display(corr)

## 2 · Incremental logistic regression — simplest first
Binomial GLM (successes `n_des` of `n_designs`). Nested ladder compared by McFadden pseudo-R², AIC and a likelihood-ratio test: is disjointedness alone already enough?

In [ ]:
# --- binomial-logistic fit wrapper (statsmodels if present, else sklearn on expanded trials) ---
def fit_glm(df, cols):
    """Return dict: predict(df)->p, coef{name:beta}, se, pval (if available), llf, k, n."""
    if not cols:                                             # intercept-only
        p0 = df.n_des.sum()/df.n_designs.sum()
        return dict(predict=lambda d: np.full(len(d), p0), coef={'const':np.log(p0/(1-p0))},
                    se={}, pval={}, llf=_ll(df, np.full(len(df),p0)), k=1, n=len(df))
    if HAVE_SM:
        X = sm.add_constant(df[cols], has_constant='add')
        y = df[['n_des']].assign(fail=df.n_designs-df.n_des)[['n_des','fail']]
        r = sm.GLM(y, X, family=sm.families.Binomial()).fit()
        return dict(predict=lambda d: r.predict(sm.add_constant(d[cols], has_constant='add')),
                    coef=r.params.to_dict(), se=r.bse.to_dict(), pval=r.pvalues.to_dict(),
                    llf=r.llf, k=len(cols)+1, n=len(df), sm=r)
    # sklearn fallback: expand each motif into successes/failures with sample weights
    X = np.vstack([df[cols].values, df[cols].values])
    yb = np.r_[np.ones(len(df)), np.zeros(len(df))]
    w  = np.r_[df.n_des.values, (df.n_designs-df.n_des).values]; mk = w>0
    lr = LogisticRegression(penalty=None, max_iter=3000).fit(X[mk], yb[mk], sample_weight=w[mk])
    coef = {'const': float(lr.intercept_[0])}; coef.update(dict(zip(cols, lr.coef_[0])))
    pr = lambda d: lr.predict_proba(d[cols].values)[:,1]
    return dict(predict=pr, coef=coef, se={}, pval={}, llf=_ll(df, pr(df)), k=len(cols)+1, n=len(df), lr=lr)

def _ll(df, p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return float((df.n_des*np.log(p) + (df.n_designs-df.n_des)*np.log(1-p)).sum())

from scipy.stats import chi2
LADDER = {'M0 intercept':[], 'M1 +disjoint':['disjointed_parts_z'],
          'M2 +length':['disjointed_parts_z','motif_length_z'],
          'M3 +len*disj':['disjointed_parts_z','motif_length_z','len_x_disj'],
          'M4 +SSE,RSA':['disjointed_parts_z','motif_length_z','len_x_disj','H','E','mean_rsa_z']}
rows = []
for meth, df in tab.groupby('method'):
    ll0 = fit_glm(df, [])['llf']; prev = None
    for name, cols in LADDER.items():
        ft = fit_glm(df, cols)
        r2 = 1 - ft['llf']/ll0 if ll0 else np.nan
        aic = 2*ft['k'] - 2*ft['llf']
        lrt = '' if prev is None else f"{1-chi2.cdf(2*(ft['llf']-prev['llf']), ft['k']-prev['k']):.1e}"
        rows.append(dict(method=meth, model=name, pseudoR2=round(r2,3), AIC=round(aic,1), LRT_p=lrt))
        prev = ft
lad = pd.DataFrame(rows)
display(lad.pivot(index='method', columns='model', values='pseudoR2'))
print('AIC / LRT-p vs previous model:'); display(lad.pivot(index='method', columns='model', values='AIC'))

**Read-off:** M1 (disjointedness alone) captures nearly all the explainable variance for every method except La-Proteina *unindexed*, which is largely insensitive to disjointedness (it places fragments freely). M4 is the deployed model (adds SSE + burial).

In [ ]:
# --- deployed model = M4; coefficient table (odds ratios) per method ---
FINAL = LADDER['M4 +SSE,RSA']
FIT = {}
coef_rows = []
for meth, df in tab.groupby('method'):
    ft = fit_glm(df, FINAL); FIT[meth] = ft
    for name in ['const']+FINAL:
        b = ft['coef'].get(name, np.nan)
        coef_rows.append(dict(method=meth, term=name, beta=round(b,3),
                              odds_ratio=round(np.exp(b),3),
                              p=(round(ft['pval'].get(name,np.nan),3) if ft['pval'] else np.nan)))
coefs = pd.DataFrame(coef_rows)
display(coefs.pivot(index='term', columns='method', values='beta')
        .reindex(['const']+FINAL))

## 2.5 · Predictive capability — cross-validated AUC, calibration, predicted vs observed
How well the model generalizes to an unseen motif (grouped 5-fold CV), whether probabilities are calibrated, and per-method fit quality.

In [ ]:
# --- grouped 5-fold CV (motif-level ">=1 designable") : AUC + Brier + calibration ---
cvrows, oof = [], {}
for meth, df in tab.groupby('method'):
    df = df.reset_index(drop=True); ybin = (df.n_des>0).astype(int)
    p = np.full(len(df), float(ybin.mean()))
    if ybin.nunique()==2:
        for tr,te in GroupKFold(n_splits=5).split(df, ybin, groups=df.task):
            ft = fit_glm(df.iloc[tr], FINAL); p[te] = np.clip(np.asarray(ft['predict'](df.iloc[te])), 1e-6, 1)
        auc = roc_auc_score(ybin, p); brier = brier_score_loss(ybin, p)
    else:
        auc = brier = np.nan
    oof[meth] = (ybin.values, p)
    cvrows.append(dict(method=meth, AUC=round(auc,3), Brier=round(brier,3), base_rate=round(ybin.mean(),2)))
cvdf = pd.DataFrame(cvrows).set_index('method'); display(cvdf)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]; order = cvdf.AUC.sort_values().index
ax.barh([LABELS[m] for m in order], cvdf.loc[order,'AUC'], color=[PALETTE[m] for m in order],
        edgecolor='black', lw=0.7)
for i,m in enumerate(order): ax.text(cvdf.loc[m,'AUC']+.005, i, f"{cvdf.loc[m,'AUC']:.2f}", va='center', fontsize=8)
ax.axvline(0.5, color='k', ls=':'); ax.set_xlim(0.5, 1.0)
ax.set_xlabel('cross-validated AUC (motif-level designable)')
ax.set_title('Predictive capability (5-fold grouped CV)', fontweight='bold'); prettify(ax)

ax = axes[1]                                                # pooled calibration
allp = np.concatenate([oof[m][1] for m in oof]); ally = np.concatenate([oof[m][0] for m in oof])
bins = np.linspace(0, 1, 9); bi = np.clip(np.digitize(allp, bins)-1, 0, len(bins)-2)
xs, ys = [], []
for b in range(len(bins)-1):
    mk = bi==b
    if mk.sum() > 20: xs.append(allp[mk].mean()); ys.append(ally[mk].mean())
ax.plot([0,1],[0,1],'k--', lw=.8); ax.plot(xs, ys, 'o-', color='#e71d36')
ax.set_xlabel('predicted probability'); ax.set_ylabel('observed fraction designable')
ax.set_title('Calibration (pooled, CV predictions)', fontweight='bold'); prettify(ax)
plt.tight_layout(); plt.show()

In [ ]:
# --- predicted p_hat vs OBSERVED per-motif success rate, one panel per method ---
from scipy.stats import spearmanr
pv = add_design_cols(tab).copy(); pv['p_hat'] = np.nan
for meth in pv.method.unique():
    mk = pv.method==meth; pv.loc[mk,'p_hat'] = np.asarray(FIT[meth]['predict'](pv[mk]))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, meth in zip(axes.ravel(), LOAD):
    s = pv[pv.method==meth]
    if not len(s): ax.axis('off'); continue
    ax.scatter(s.p_hat, s.succ_rate, s=14, alpha=0.45, color=PALETTE[meth], edgecolor='none')
    mx = max(float(s.p_hat.max()), float(s.succ_rate.max()))*1.05 or 1.0
    ax.plot([0,mx],[0,mx],'k--', lw=.8); ax.set_xlim(0,mx); ax.set_ylim(0,mx)
    r2  = 1 - ((s.succ_rate-s.p_hat)**2).sum()/((s.succ_rate-s.succ_rate.mean())**2).sum()
    rho = spearmanr(s.p_hat, s.succ_rate).correlation
    ax.set_title(f'{LABELS[meth]}   R²={r2:.2f}  ρ={rho:.2f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('predicted p̂'); ax.set_ylabel('observed success rate'); prettify(ax)
plt.suptitle('Predictive capability — predicted vs observed per-motif success rate', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# --- matrix comparison of the methods: coefficients + observed success by disjointedness ---
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
# (A) logistic coefficients (log-odds):  method x feature
cm = pd.DataFrame({m:{f:FIT[m]['coef'].get(f,0.0) for f in FINAL} for m in LOAD}).T.reindex(list(LOAD))
ax = axes[0]; im = ax.imshow(cm.values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
ax.set_xticks(range(len(FINAL))); ax.set_xticklabels(FINAL, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(cm))); ax.set_yticklabels([LABELS[m] for m in cm.index])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): ax.text(j, i, f'{cm.values[i,j]:+.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Logistic coefficients (log-odds) · method × feature', fontweight='bold'); fig.colorbar(im, ax=ax, shrink=.8)
# (B) observed success rate:  method x disjointedness bin
tab['disj_bin'] = pd.cut(tab.disjointed_parts, [0,1,2,3,99], labels=['1','2','3','4+'])
sm = tab.groupby(['method','disj_bin']).succ_rate.mean().unstack().reindex(list(LOAD))
ax = axes[1]; im = ax.imshow(sm.values, cmap='viridis', aspect='auto', vmin=0)
ax.set_xticks(range(sm.shape[1])); ax.set_xticklabels(sm.columns)
ax.set_yticks(range(len(sm))); ax.set_yticklabels([LABELS[m] for m in sm.index])
for i in range(sm.shape[0]):
    for j in range(sm.shape[1]):
        v = sm.values[i,j]
        if v==v: ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='white')
ax.set_xlabel('disjointedness (# fragments)')
ax.set_title('Observed success rate · method × disjointedness', fontweight='bold'); fig.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout(); plt.show()

## 3 · Per-feature contribution (nomogram-style)
For a motif, contribution of feature *j* = `beta_j * (x_j - mean_j)` in log-odds; bars + intercept sum to `logit(p)`. Shows exactly which feature pushed the probability which way.

In [ ]:
# --- log-odds contribution decomposition (exact for a logistic model) ---
def contributions(meth, row):
    ft = FIT[meth]; c = ft['coef']; out = {'intercept': c.get('const',0.0)}
    for name in FINAL:
        xz = row[name] if name in ('H','E') else row.get(name, np.nan)
        out[name] = c.get(name,0.0) * float(xz)              # x already centered (z) or 0/1 dummy
    logit = sum(out.values()); return out, 1/(1+np.exp(-logit))

def plot_contrib(task, meth, ax=None):
    row = add_design_cols(tab[(tab.task==task)&(tab.method==meth)]).iloc[0]
    parts, p = contributions(meth, row)
    items = [(k,v) for k,v in parts.items() if k!='intercept']
    items.sort(key=lambda kv: abs(kv[1]))
    if ax is None: fig, ax = plt.subplots(figsize=(6,3.2))
    ax.barh([k for k,_ in items], [v for _,v in items],
            color=['#2a9d8f' if v>0 else '#e76f51' for _,v in items], edgecolor='black', lw=.5)
    ax.axvline(0, color='k', lw=.8)
    ax.set_title(f'{task} · {LABELS[meth]}  →  p̂ = {p:.2f}  (obs {row.succ_rate:.2f})', fontsize=10, fontweight='bold')
    ax.set_xlabel('log-odds contribution'); prettify(ax); return p

# example: a disjointed motif across methods
ex = tab[tab.disjointed_parts>=3].sort_values('n_designs',ascending=False).task.iloc[0]
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, meth in zip(axes.ravel(), LOAD):
    if ((tab.task==ex)&(tab.method==meth)).any(): plot_contrib(ex, meth, ax)
    else: ax.axis('off')
plt.suptitle(f'Per-feature contribution for {ex} (disjointed) across methods', fontweight='bold')
plt.tight_layout(); plt.show()

## 4 · ROI — GPU-hours per designable design
Decision metric = `gpuh_per_design / p̂` (expected compute to obtain one lab-testable design). Also `P(≥1 in N) = 1-(1-p̂)^N`.

In [ ]:
# --- ROI table: predicted p, expected GPU-h per designable design, P(>=1 in N) ---
def predict_all(df):
    o = add_design_cols(df).copy()
    o['p_hat'] = np.nan
    for meth in o.method.unique():
        mk = o.method==meth
        o.loc[mk,'p_hat'] = np.clip(FIT[meth]['predict'](o[mk]), 1e-6, 1)
    o['gpuh_per_success'] = o.gpuh_per_design / o.p_hat
    o['p_ge1_in_100'] = 1-(1-o.p_hat)**100
    return o
roi = predict_all(tab)

# method ranking by ROI, split by disjointedness regime
roi['regime'] = np.where(roi.disjointed_parts>1, 'disjoint (>1)', 'contiguous (1)')
summ = (roi.groupby(['regime','method'])
        .agg(p_hat=('p_hat','mean'), gpuh_per_success=('gpuh_per_success','median'))
        .round(3).sort_values(['regime','gpuh_per_success']))
display(summ)

fig, ax = plt.subplots(figsize=(9,4.5))
for reg, mk in [('contiguous (1)','o'), ('disjoint (>1)','s')]:
    s = summ.loc[reg]
    ax.scatter(s.gpuh_per_success, s.p_hat, s=90, marker=mk, edgecolor='black',
               c=[PALETTE[m] for m in s.index], label=reg, zorder=3)
    for m,r in s.iterrows(): ax.annotate(LABELS[m], (r.gpuh_per_success, r.p_hat), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.set_xscale('log'); ax.set_xlabel('median GPU-h per designable design  (lower = better ROI)')
ax.set_ylabel('mean predicted success p̂'); ax.legend(); prettify(ax)
ax.set_title('ROI by method and disjointedness regime', fontweight='bold'); plt.tight_layout(); plt.show()

## 5 · Gap-fill what-if + guideline tree (the rulebook)
Does filling gaps (base→filled, fewer fragments) pay off? Then a depth-3 decision tree that maps motif features → the **best-ROI method** — the deployable rulebook. Expect the split on disjointedness: contiguous → cheapest method (Protpardelle); disjoint → the disjointedness-robust one (La-Proteina unindexed).

In [ ]:
# --- base -> filled Δp̂ (fill gaps to cut disjointedness), per method ---
piv = roi.pivot_table(index=['pdb_name','method'], columns='variant', values='p_hat')
if {'base','filled'}.issubset(piv.columns):
    piv = piv.dropna(subset=['base','filled']); piv['delta'] = piv.filled - piv.base
    g = piv.groupby('method').delta.agg(['mean','median','count']).round(3)
    print('gap-fill Δp̂ (filled − base), matched patches:'); display(g)
else:
    print('no matched base/filled pairs with both predictions')

# --- guideline tree: which METHOD gives the best ROI, as a function of motif features ---
rec = roi.loc[roi.groupby('task').gpuh_per_success.idxmin()].copy()   # best-ROI method per motif
print('ROI-recommended method (min GPU-h per designable design):')
display(rec.method.value_counts().to_frame('n_motifs'))
print('by disjointedness regime:')
display(pd.crosstab(rec.regime, rec.method.map(LABELS)))
Xtree = rec[['disjointed_parts','motif_length','mean_rsa']].values
ytree = rec.method.values
dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=0).fit(Xtree, ytree)
print(export_text(dt, feature_names=['disjoint_parts','motif_length','mean_rsa']))
fig, ax = plt.subplots(figsize=(13,6))
plot_tree(dt, feature_names=['disjoint_parts','motif_length','mean_rsa'],
          class_names=[LABELS.get(c,c) for c in dt.classes_], filled=True, rounded=True, fontsize=8, ax=ax)
plt.title('Guideline tree — which method to use (best ROI) from motif features', fontweight='bold'); plt.show()

## 6 · Nearest library analog (optional sanity check)
For a query motif, the k nearest library motifs in standardized feature space and their observed per-method success.

In [ ]:
# --- nearest-neighbour empirical analog (descriptor space) ---
from sklearn.neighbors import NearestNeighbors
FEAT_NN = ['disjointed_parts','motif_length','mean_rsa','rg','linearity','planarity','mean_cx']
uu = tab.drop_duplicates('task').dropna(subset=FEAT_NN).reset_index(drop=True)
Z = (uu[FEAT_NN]-uu[FEAT_NN].mean())/uu[FEAT_NN].std(ddof=0)
nn = NearestNeighbors(n_neighbors=6).fit(Z.values)
def analog(task, k=5):
    i = uu.index[uu.task==task][0]
    d,idx = nn.kneighbors(Z.values[i:i+1], n_neighbors=k+1)
    nb = uu.iloc[idx[0][1:]].task.tolist()
    obs = (tab[tab.task.isin(nb)].groupby('method').succ_rate.mean().round(3))
    print(f'query {task}: nearest {nb}'); display(obs.to_frame('neighbour_mean_succ'))
analog(uu.task.iloc[0])

## 7 · Export the fitted model for `predict_motif.py`
Coefficients + feature means/SDs + per-method cost → one JSON the CLI loads to score a brand-new motif.

In [ ]:
# --- export success_model.json (consumed by scripts/predict_motif.py) ---
model = dict(final_features=FINAL, cont=CONT, mu=MU, sd=SD,
             gates=dict(motifRMSD=MOTIF_RMSD_MAX, scRMSD=SC_RMSD_MAX, ranking_score=RANK_MIN),
             nsamples=NSAMPLES, refold_gpuh_per_design=REFOLD_GPUH_PER_DESIGN,
             methods={})
for meth, ft in FIT.items():
    model['methods'][meth] = dict(coef=ft['coef'],
        gpuh_per_design=float(tab[tab.method==meth].gpuh_per_design.iloc[0]),
        label=LABELS[meth])
out = f'{OUTDIR}/success_model.json'
with open(out,'w') as fh: json.dump(model, fh, indent=1)
print('wrote', out, '\nmethods:', list(model['methods']))

## 8 · Apply the model to an EXTERNAL / unseen motif set (+ validate vs Lisa's 2mhs)

**What you feed it:** a *feature* CSV — one row per motif with `motif_num, motif_length,
disjointed_parts, dominant_sse, mean_rsa`. The raw motif-**definition** CSV
(`lbrandenburg/motif_scaffolding/2mhs_c_motif.csv`, surface residue + contacts) is **not** enough — the
model needs the extracted *descriptors*. Generate them with the same `esp` extractor the library used:

```
source /cluster/work/wicky/bekessel/gen_env/bin/activate
export PYTHONPATH=/cluster/work/wicky/bekessel/epitope_scaffolding/pipeline
python -m esp.run extract-motifs --config library/configs/mcl1.yaml --out /cluster/work/wicky/bekessel/motif_library --archive
# -> writes mcl1_c_motif.csv WITH motif_length / disjointed_parts / dominant_sse / mean_rsa (motif_num = surface residue)
```

**2mhs is a genuine held-out test:** mcl1 (= 2mhs) is *not* in this benchmark pool (`..._pad5.csv` has
only her2/tnfa/il1b/il2/pdl1/il13), so the model never saw it. Lisa ran RFdiffusion at **1000 traj**, so
her per-motif designable rate is a high-N ground truth. We predict rfdiff p̂ from the descriptors and
correlate against her observed rate (join on `motif_num`; her numbering == surface residue == our esp
`motif_num`). Same pattern for any new interface: extract descriptors → predict → (optionally) validate.
Batch CLI equivalent: `python scripts/predict_motif.py --batch mcl1_c_motif.csv --out preds.csv`.

In [ ]:
# --- external prediction + optional Lisa-2mhs validation (all paths are knobs) ---
EXT_FEATURES = f'{LIB}/mcl1_c_motif.csv'   # <- target FEATURE csv (esp extract-motifs output; has descriptors)
EXT_QUERYPDB = None            # pdb_name filter; None = all rows in the file
EXT_METHOD   = 'rfdiff'        # method to score / compare (Lisa only ran rfdiff)
EXT_OBSERVED = '/cluster/work/wicky/bekessel/analysis/reports/af3_refold_2mhs_c_motif.csv'  # Lisa's, or None

if not os.path.exists(EXT_FEATURES):
    print(f'No feature csv at {EXT_FEATURES}. Generate it first (see the markdown above):\n'
          '  python -m esp.run extract-motifs --config library/configs/mcl1.yaml --out '+LIB+' --archive')
    ext = pd.DataFrame()
else:
    ext = pd.read_csv(EXT_FEATURES)
    if EXT_QUERYPDB and 'pdb_name' in ext: ext = ext[ext.pdb_name==EXT_QUERYPDB]
    if 'variant' in ext: ext = ext[ext.variant=='base']     # one row per patch
    need = ['motif_num','motif_length','disjointed_parts','dominant_sse','mean_rsa']
    ext = ext.dropna(subset=[c for c in need if c in ext]).copy()

if len(ext):
    extd = add_design_cols(ext)
    extd['p_hat'] = np.clip(np.asarray(FIT[EXT_METHOD]['predict'](extd)), 1e-6, 1)
    extd['gpuh_per_success'] = tab[tab.method==EXT_METHOD].gpuh_per_design.iloc[0] / extd.p_hat
    print(f'predicted {EXT_METHOD} for {len(extd)} external motifs | mean p̂ = {extd.p_hat.mean():.3f}')
    display(extd[['motif_num','motif_length','disjointed_parts','dominant_sse','mean_rsa','p_hat']]
            .round(3).sort_values('p_hat').head(12))

    if EXT_OBSERVED and os.path.exists(EXT_OBSERVED):
        from scipy.stats import spearmanr
        obs = pd.read_csv(EXT_OBSERVED)
        obs['sp'] = (((obs.motifRMSD<=MOTIF_RMSD_MAX)&(obs.scRMSD<=SC_RMSD_MAX)).astype(int)
                     if 'motifRMSD' in obs else obs.get('designable',0))
        og = obs.groupby('motif_num').sp.mean().rename('obs_rate').reset_index()
        j = extd.assign(motif_num=extd.motif_num.astype(int)).merge(og, on='motif_num', how='inner')
        if len(j) >= 5:
            rho = spearmanr(j.p_hat, j.obs_rate).correlation
            fig, ax = plt.subplots(figsize=(5.6,5))
            ax.scatter(j.p_hat, j.obs_rate, s=32, color=PALETTE[EXT_METHOD], edgecolor='black', lw=.4)
            mx = max(float(j.p_hat.max()), float(j.obs_rate.max()))*1.1
            ax.plot([0,mx],[0,mx],'k--', lw=.8)
            ax.set_xlabel(f'predicted {EXT_METHOD} p̂'); ax.set_ylabel('observed designable rate (Lisa 2mhs, 1000-traj)')
            ax.set_title(f'External validation on 2mhs · ρ={rho:.2f} (n={len(j)})', fontweight='bold'); prettify(ax)
            plt.tight_layout(); plt.show()
            print(f'Spearman predicted vs observed = {rho:.3f} over {len(j)} shared motifs '
                  '(CAVEAT: Lisa 2-gate + her numbering vs our mcl1 descriptors).')
        else:
            print(f'only {len(j)} motif_num overlaps — check numbering (Lisa 2mhs vs our mcl1 pool)')
    else:
        print('EXT_OBSERVED not found — predictions only. Point it at Lisa 2mhs af3_refold csv to validate.')